# <p align="center"> Get Ligand PDBs from Prot-Frag PDBs </p>

### Load Targeted Gemmi.Residues into a Gemmi.Structure and Save

In [ ]:
import gemmi
def create_ligand_file( residueSpan, pdb_name = None, save_path = None):

    st = gemmi.Structure()
    
    if pdb_name: st.name = pdb_name
    else: st.name = "ligand"

    model = gemmi.Model(1)

    chain = gemmi.Chain( "A")

    for residue in residueSpan:
        chain.add_residue( residue)

    model.add_chain( chain )
    st.add_model( model )

    if save_path:
        st.write_pdb(save_path, gemmi.PdbWriteOptions() )
    pass

### Extract for a given Gemmi Prot-Frag Obj

In [ ]:
import pathlib
import gemmi
def extractLigands( pdb_st: gemmi.Structure, saveDirPath: pathlib.Path, protName: str ):
    """
    Extracts ligands from a PDB structure and saves them as individual PDB files.
    Args:
    - pdb_st (gemmi.Structure): The PDB structure object containing the ligands.
    - saveDirPath (pathlib.Path): The directory path where the ligand PDB files
    - protName (str): The name of the protein, used to name the ligand PDB files.
    Returns:
    - None: The function saves the ligand PDB files to the specified directory.
    """

    saveDirPath.mkdir( parents=True, exist_ok=True)
    # record_log = {}
    # ligands_log = {"Ligands Count" : 0, "Ligand Names" : []}
    if len(pdb_st) == 1:
        model = pdb_st[0]
        for chain in model:
            # if chain.get_polymer():
            #     print(chain.name)
            #     resSpan = chain.whole()
            #     print( resSpan)
            if chain.get_ligands():
                # ligands_log["Ligands Count"] += 1
                print( f"Processing Chain: {chain.name }")
                resSpan =  chain.whole() 
                ligName = list( set( resSpan.extract_sequence() ) )
                if len(ligName) > 1:
                    print( f"Bad arrangement of ligands with more than one in a chain: {ligName}" )
                    for lig in ligName:
                        ligPDBName = f"{protName}_{lig}.pdb"

                        for residue in chain.whole():
                            print(residue.name)
                            # Continue code

                else:
                    print( f"Processing Ligand: {ligName[0]}")
                    ligPDBName = f"{protName}_{ligName[0]}.pdb"
                    saveFilePath = saveDirPath / ligPDBName
                    saveFilePath = saveFilePath.resolve().as_posix().__str__()
                    # print( type(saveFilePath) )
                    create_ligand_file( resSpan, ligPDBName, saveFilePath)

                # ligands_log["Ligand Names"].append( ligName )

    elif len(pdb_st) == 0:
        print( "Error with Model")
    else:
        print("More than one model")

### Extract for all proteins

In [ ]:
from pathlib import Path
import glob
import gemmi

# Initialize notebook environment
import sys
from pathlib import Path
sys.path.append( Path("../..").resolve().absolute().__str__() )

saveDirPath = Path( "../../../data/s3Data/02-ligandPDBs")
for path in glob.glob("../../../data/s3Data/01-refinedBoundPDBs/*.pdb"):
    boundProtPath = Path(path)
    print(boundProtPath.resolve())
    protName = boundProtPath.name[:-4]
    print(f"Protein Name: {protName}")
    pdb = gemmi.read_pdb(boundProtPath.resolve().as_posix() )
    pdb.setup_entities()
    pdb.assign_label_seq_id()
    extractLigands( pdb, saveDirPath, protName )


### Creating .txt files with paths to load into PyMOL

In [ ]:
import glob
from pathlib import Path
lst_paths = glob.glob("../../../data/s3Data/02-ligandPDBs/NUDT7A*LIG.pdb")

pathsLst = Path("../../../../PyMOL/load_NUDT7A_LIG.txt").resolve()
with open( pathsLst , 'w') as file:
    for path in lst_paths:
        pathStr = Path(path).resolve().as_posix() 
        ligNumb = Path(path).name[8:-4] 
        file.write(f"{pathStr}   {ligNumb}\n")

## I.e.

In [ ]:
from xaidar.expDataAnalysis.dataExtract import extractAllLigands, prepPyMOLVisualization

saveDirPath = Path( "../../../data/s3Data/02-ligandPDBs")
lst_pdbPaths = glob.glob("../../../data/s3Data/01-refinedBoundPDBs/*.pdb")
extractAllLigands( lst_pdbPaths,saveDirPath=saveDirPath)

lst_paths = glob.glob("../../../data/s3Data/02-ligandPDBs/NUDT7A*LIG.pdb")
filePath = Path("../../../../PyMOL/load_NUDT7A_LIG.txt").resolve()
prepPyMOLVisualization( filePath, lst_paths )